In [15]:
# check files in county_health_rankins

import os
print(os.getcwd())


files = os.listdir('../data/raw/county_health_rankings')
for f in sorted(files):
    print(f)

/Users/douglas/Documents/GitHub/SDOH_Housing_Health_CA/notebooks
2016 County Health Rankings California Data - v3.xls
2017 County Health Rankings California Data - v2.xls
2018 County Health Rankings California Data - v3.xls
2019 County Health Rankings California Data - v1_0.xls
2020 County Health Rankings California Data - v1_0.xlsx
2021 County Health Rankings California Data - v1.xlsx
2022 County Health Rankings California Data - v2.xlsx
2023 County Health Rankings California Data - v3.xlsx
2024 County Health Rankings California Data - v2.xlsx
2025 County Health Rankings California Data - v3.xlsx


In [2]:
import pandas as pd
import numpy as np
import os

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

Load the 2024 CHR file to inspect structure and confirm availability of outcome 
variables (Premature Death, Preventable Hospital Stays, Poor Mental Health Days) 
that we will later regress on Median Multiple, our measure of housing unaffordability.

In [3]:
# Load 2024 CHR file
file_path = '../data/raw/county_health_rankings/2024 County Health Rankings California Data - v2.xlsx'

df_2024 = pd.read_excel(
    file_path,
    sheet_name='Additional Measure Data',
    header=[0, 1]  # Multi-row header based on what we saw
)

# Inspect
print('Shape:', df_2024.shape)
print('\nColumn names:')
for col in df_2024.columns:
    print(col)

Shape: (59, 344)

Column names:
('Unnamed: 0_level_0', 'FIPS')
('Unnamed: 1_level_0', 'State')
('Unnamed: 2_level_0', 'County')
('Life Expectancy', 'Life Expectancy')
('Life Expectancy', '95% CI - Low')
('Life Expectancy', '95% CI - High')
('Life Expectancy', 'Life Expectancy (Hispanic (all races))')
('Life Expectancy', 'Life Expectancy (Hispanic (all races)) 95% CI - Low')
('Life Expectancy', 'Life Expectancy (Hispanic (all races)) 95% CI - High')
('Life Expectancy', 'Life Expectancy (Non-Hispanic AIAN)')
('Life Expectancy', 'Life Expectancy (Non-Hispanic AIAN) 95% CI - Low')
('Life Expectancy', 'Life Expectancy (Non-Hispanic AIAN) 95% CI - High')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Asian)')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Asian) 95% CI - Low')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Asian) 95% CI - High')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Black)')
('Life Expectancy', 'Life Expectancy (Non-Hispanic Black) 95% CI - Low')
('L

Of the potential outcome variables available in the 'Additional Measure Data' 
sheet, Life Expectancy was considered but rejected in favor of Premature Death 
(YPLL — Years of Potential Life Lost), available in the 'Select Measure Data' 
sheet. YPLL weights early deaths more heavily than average life expectancy, 
making it more sensitive to the effects of housing unaffordability and economic 
stress on working-age adults.

From the 'Select Measure Data' sheet, three outcome variables were selected 
for regression against the Median Multiple, our primary measure of housing 
unaffordability: Premature Death (Years of Potential Life Lost Rate), 
Preventable Hospital Stays (Preventable Hospitalization Rate), and Poor 
Mental Health Days (Average Number of Mentally Unhealthy Days). Two additional 
variables — Children in Poverty and Low Birthweight — were identified as 
potential mediators for subsequent mediation analysis.


In [4]:
df_select = pd.read_excel(
    file_path,
    sheet_name='Select Measure Data',
    header=[0, 1]
)

print('Shape:', df_select.shape)
print('\nColumn names:')
for col in df_select.columns:
    print(col)

Shape: (59, 272)

Column names:
('Unnamed: 0_level_0', 'FIPS')
('Unnamed: 1_level_0', 'State')
('Unnamed: 2_level_0', 'County')
('Premature Death', 'Unreliable')
('Premature Death', 'Deaths')
('Premature Death', 'Years of Potential Life Lost Rate')
('Premature Death', '95% CI - Low')
('Premature Death', '95% CI - High')
('Premature Death', 'National Z-Score')
('Premature Death', 'YPLL Rate (Hispanic (all races))')
('Premature Death', 'YPLL Rate (Hispanic (all races)) 95% CI - Low')
('Premature Death', 'YPLL Rate (Hispanic (all races)) 95% CI - High')
('Premature Death', 'YPLL Rate (Hispanic (all races)) Unreliable')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN)')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) 95% CI - Low')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) 95% CI - High')
('Premature Death', 'YPLL Rate (Non-Hispanic AIAN) Unreliable')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian)')
('Premature Death', 'YPLL Rate (Non-Hispanic Asian) 95% CI - Low')
('Pre

In [5]:
df_select_outcomes = df_select[[('Unnamed: 0_level_0', 'FIPS'),('Unnamed: 2_level_0', 'County'),
('Premature Death', 'Years of Potential Life Lost Rate'), ('Poor Mental Health Days', 'Average Number of Mentally Unhealthy Days'),
('Preventable Hospital Stays', 'Preventable Hospitalization Rate')]]

print(df_select_outcomes.head(3).to_string())

  Unnamed: 0_level_0 Unnamed: 2_level_0                   Premature Death                   Poor Mental Health Days       Preventable Hospital Stays
                FIPS             County Years of Potential Life Lost Rate Average Number of Mentally Unhealthy Days Preventable Hospitalization Rate
0               6000                NaN                       6373.194549                                  4.650926                             2153
1               6001            Alameda                       4981.213394                                  5.070780                             2102
2               6003             Alpine                               NaN                                  5.541764                              921


In [6]:
df_select_mediators = df_select[[('Unnamed: 0_level_0', 'FIPS'),('Unnamed: 2_level_0', 'County'), ('Children in Poverty', '% Children in Poverty'),('Low Birthweight', '% Low Birthweight')]]

print(df_select_mediators)
print(df_select_mediators.head(3).to_string())


   Unnamed: 0_level_0 Unnamed: 2_level_0   Children in Poverty  \
                 FIPS             County % Children in Poverty   
0                6000                NaN                  15.3   
1                6001            Alameda                  10.3   
2                6003             Alpine                  26.6   
3                6005             Amador                  12.0   
4                6007              Butte                  18.7   
5                6009          Calaveras                  16.2   
6                6011             Colusa                  13.7   
7                6013       Contra Costa                  10.0   
8                6015          Del Norte                  22.5   
9                6017          El Dorado                   6.5   
10               6019             Fresno                  25.1   
11               6021              Glenn                  19.2   
12               6023           Humboldt                  20.5   
13        

In [7]:
# drop the state row
df_select_outcomes = df_select_outcomes.drop(index=0)
df_select_mediators = df_select_mediators.drop(index=0)

# check for null values
print(df_select_outcomes.isna().sum())
print(df_select_mediators.isna().sum())

Unnamed: 0_level_0          FIPS                                         0
Unnamed: 2_level_0          County                                       0
Premature Death             Years of Potential Life Lost Rate            2
Poor Mental Health Days     Average Number of Mentally Unhealthy Days    0
Preventable Hospital Stays  Preventable Hospitalization Rate             0
dtype: int64
Unnamed: 0_level_0   FIPS                     0
Unnamed: 2_level_0   County                   0
Children in Poverty  % Children in Poverty    0
Low Birthweight      % Low Birthweight        1
dtype: int64


In [8]:
# check counties impacted by nulls
df_select_outcomes[df_select_outcomes[('Premature Death', 'Years of Potential Life Lost Rate')].isna()]



,Unnamed: 0_level_0,Unnamed: 2_level_0,Premature Death,Poor Mental Health Days,Preventable Hospital Stays
,FIPS,County,Years of Potential Life Lost Rate,Average Number of Mentally Unhealthy Days,Preventable Hospitalization Rate
2,6003,Alpine,NaN,5.541764,921
46,6091,Sierra,NaN,5.482310,1644


In [9]:
df_select_mediators[df_select_mediators[('Low Birthweight', '% Low Birthweight')].isna()]

,Unnamed: 0_level_0,Unnamed: 2_level_0,Children in Poverty,Low Birthweight
,FIPS,County,% Children in Poverty,% Low Birthweight
2,6003,Alpine,26.6,NaN


Null values were observed in two rural counties: Alpine (FIPS 06003) and 
Sierra (FIPS 06091). All county data will be retained in order to preserve 
valid observations for other outcome variables. Null values will be excluded 
listwise at the model level rather than removed from the dataset. This 
approach acknowledges unavoidable data suppression in low-population counties 
as a limitation of the analysis.

In [10]:
# revisit after column naming conventions are understood


# # relevent_variables = ['Unnamed: 0_level_0', 'FIPS','Unnamed: 2_level_0', 'County', 'Premature Death', 'Years of Potential Life Lost Rate', 'Poor Mental Health Days', 'Average Number of Mentally Unhealthy Days',
# # 'Preventable Hospital Stays', 'Preventable Hospitalization Rate','Children in Poverty', '% Children in Poverty','Low Birthweight', '% Low Birthweight']
   
# for f in files:
#     if '2025' in f:
#         continue
#     file_path = '../data/raw/county_health_rankings/' + f
#     df_year = pd.read_excel(
#         file_path,
#         sheet_name='Select Measure Data',
#         header=[0, 1])
#     print('File:', f, '| Shape:', df_year.shape)     



In [11]:
for f in files:
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    xl = pd.ExcelFile(file_path)
    print(f, '|', xl.sheet_names)

2023 County Health Rankings California Data - v3.xlsx | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measure Sources & Years']
2018 County Health Rankings California Data - v3.xls | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measure Sources & Years']
2017 County Health Rankings California Data - v2.xls | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measure Sources & Years']
2020 County Health Rankings California Data - v1_0.xlsx | ['Introduction', 'Outcomes & Factors Rankings', 'Outcomes & Factors SubRankings', 'Ranked Measure Data', 'Additional Measure Data', 'Ranked Measure Sources & Years', 'Addtl Measur

In [12]:
# Load 2016 CHR file
file_path = '../data/raw/county_health_rankings/2016 County Health Rankings California Data - v3.xls'

df_2016 = pd.read_excel(
    file_path,
    sheet_name='Ranked Measure Data',
    header=[0, 1]  # Multi-row header based on what we saw
)

# Inspect
print('Shape:', df_2016.shape)
print('\nColumn names:')
for col in df_2016.columns:
    print(col)

Shape: (61, 156)

Column names:
('Unnamed: 0_level_0', 'FIPS')
('Unnamed: 1_level_0', 'State')
('Unnamed: 2_level_0', 'County')
('Premature death', '# Deaths')
('Premature death', 'Years of Potential Life Lost Rate')
('Premature death', '95% CI - Low')
('Premature death', '95% CI - High')
('Premature death', 'Z-Score')
('Poor or fair health', '% Fair/Poor')
('Poor or fair health', '95% CI - Low')
('Poor or fair health', '95% CI - High')
('Poor or fair health', 'Z-Score')
('Poor physical health days', 'Physically Unhealthy Days')
('Poor physical health days', '95% CI - Low')
('Poor physical health days', '95% CI - High')
('Poor physical health days', 'Z-Score')
('Poor mental health days', 'Mentally Unhealthy Days')
('Poor mental health days', '95% CI - Low')
('Poor mental health days', '95% CI - High')
('Poor mental health days', 'Z-Score')
('Low birthweight', 'Unreliable')
('Low birthweight', '# Low Birthweight Births')
('Low birthweight', '# Live Births')
('Low birthweight', '% LBW')


In [13]:
for f in files:
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    xl = pd.ExcelFile(file_path)
    sheet = 'Select Measure Data' if '2024' in f else 'Ranked Measure Data'
    df_year = pd.read_excel(file_path, sheet_name=sheet, header=[0,1])
    unique_cols = []
    for col in df_year.columns:
        if col[0] not in unique_cols:
            unique_cols.append(col[0])
    print(f, '|', unique_cols)


2023 County Health Rankings California Data - v3.xlsx | ['Unnamed: 0_level_0', 'Unnamed: 1_level_0', 'Unnamed: 2_level_0', 'Premature Death', 'Poor or Fair Health', 'Poor Physical Health Days', 'Poor Mental Health Days', 'Low Birthweight', 'Adult Smoking', 'Adult Obesity', 'Food Environment Index', 'Physical Inactivity', 'Access to Exercise Opportunities', 'Excessive Drinking', 'Alcohol-Impaired Driving Deaths', 'Sexually Transmitted Infections', 'Teen Births', 'Uninsured', 'Primary Care Physicians', 'Dentists', 'Mental Health Providers', 'Preventable Hospital Stays', 'Mammography Screening', 'Flu Vaccinations', 'High School Completion', 'Some College', 'Unemployment', 'Children in Poverty', 'Income Inequality', 'Children in Single-Parent Households', 'Social Associations', 'Injury Deaths', 'Air Pollution - Particulate Matter', 'Drinking Water Violations', 'Severe Housing Problems', 'Driving Alone to Work', 'Long Commute - Driving Alone']
2018 County Health Rankings California Data - v

In [16]:
# check column name consistency at first part of tuple

for f in files:
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f

    sheet = 'Select Measure Data' if '2024' in f else 'Ranked Measure Data'
    df_year = pd.read_excel(file_path, sheet_name=sheet, header=[0,1])
    unique_cols = []
    for col in df_year.columns:
        if col[0] not in unique_cols:
            unique_cols.append(col[0])
    print(f, '|', unique_cols)

2023 County Health Rankings California Data - v3.xlsx | ['Unnamed: 0_level_0', 'Unnamed: 1_level_0', 'Unnamed: 2_level_0', 'Premature Death', 'Poor or Fair Health', 'Poor Physical Health Days', 'Poor Mental Health Days', 'Low Birthweight', 'Adult Smoking', 'Adult Obesity', 'Food Environment Index', 'Physical Inactivity', 'Access to Exercise Opportunities', 'Excessive Drinking', 'Alcohol-Impaired Driving Deaths', 'Sexually Transmitted Infections', 'Teen Births', 'Uninsured', 'Primary Care Physicians', 'Dentists', 'Mental Health Providers', 'Preventable Hospital Stays', 'Mammography Screening', 'Flu Vaccinations', 'High School Completion', 'Some College', 'Unemployment', 'Children in Poverty', 'Income Inequality', 'Children in Single-Parent Households', 'Social Associations', 'Injury Deaths', 'Air Pollution - Particulate Matter', 'Drinking Water Violations', 'Severe Housing Problems', 'Driving Alone to Work', 'Long Commute - Driving Alone']
2018 County Health Rankings California Data - v

In [ ]:
# check column name consistency at second part of tuple

target_categories = ['premature death', 'poor mental health days', 
                     'preventable hospital stays', 'children in poverty', 
                     'low birthweight']
   

for f in files:
    if '2025' in f:
        continue
    file_path = '../data/raw/county_health_rankings/' + f
    sheet = 'Select Measure Data' if '2024' in f else 'Ranked Measure Data'
    df_year = pd.read_excel(file_path, sheet_name=sheet, header=[0,1])
    unique_cols = []
    for col in df_year.columns:
        if col[0].lower() in target_categories:
            if col[1] not in unique_cols:
                unique_cols.append(col[1])
    print(f, '|', unique_cols)

2023 County Health Rankings California Data - v3.xlsx | ['Unreliable', 'Deaths', 'Years of Potential Life Lost Rate', '95% CI - Low', '95% CI - High', 'Z-Score', 'YPLL Rate (AIAN)', 'YPLL Rate (AIAN) 95% CI - Low', 'YPLL Rate (AIAN) 95% CI - High', 'YPLL Rate (AIAN) Unreliable', 'YPLL Rate (Asian)', 'YPLL Rate (Asian) 95% CI - Low', 'YPLL Rate (Asian) 95% CI - High', 'YPLL Rate (Asian) Unreliable', 'YPLL Rate (Black)', 'YPLL Rate (Black) 95% CI - Low', 'YPLL Rate (Black) 95% CI - High', 'YPLL Rate (Black) Unreliable', 'YPLL Rate (Hispanic)', 'YPLL Rate (Hispanic) 95% CI - Low', 'YPLL Rate (Hispanic) 95% CI - High', 'YPLL Rate (Hispanic) Unreliable', 'YPLL Rate (White)', 'YPLL Rate (White) 95% CI - Low', 'YPLL Rate (White) 95% CI - High', 'YPLL Rate (White) Unreliable', 'Average Number of Mentally Unhealthy Days', '% Low Birthweight', '% LBW (AIAN)', '% LBW (AIAN) 95% CI - Low', '% LBW (AIAN) 95% CI - High', '% LBW (Asian)', '% LBW (Asian) 95% CI - Low', '% LBW (Asian) 95% CI - High', '

Inspection of second-level tuple column names across all years reveals the 
following:

Consistent across all years:
- Premature Death: 'Years of Potential Life Lost Rate'
- Children in Poverty: '% Children in Poverty'

Inconsistent — require standardization in cleaning:
- Poor Mental Health Days: 'Mentally Unhealthy Days' (2016–2019) → 
  'Average Number of Mentally Unhealthy Days' (2020–2024)
- Preventable Hospital Stays: 'Preventable Hosp. Rate' (2016–2019) → 
  'Preventable Hospitalization Rate' (2020–2024)
- Low Birthweight: '% LBW' (2016–2019) → '% Low Birthweight' (2020–2024)

Note: capitalization of first-level tuple names also inconsistent — 
lowercase 2016–2022, title case 2023–2024. Both issues to be resolved 
in cleaning notebook.

In [21]:
# view income data files

print(os.getcwd())


files = os.listdir('../data/raw/census_acs')
for f in sorted(files):
    print(f)

/Users/douglas/Documents/GitHub/SDOH_Housing_Health_CA/notebooks
.DS_Store
ACSST1Y2016.S1901-Column-Metadata.csv
ACSST1Y2016.S1901-Data.csv
ACSST1Y2017.S1901-Column-Metadata.csv
ACSST1Y2017.S1901-Data.csv
ACSST1Y2018.S1901-Column-Metadata.csv
ACSST1Y2018.S1901-Data.csv
ACSST1Y2019.S1901-Column-Metadata.csv
ACSST1Y2019.S1901-Data.csv
ACSST1Y2021.S1901-Column-Metadata.csv
ACSST1Y2021.S1901-Data.csv
ACSST1Y2022.S1901-Column-Metadata.csv
ACSST1Y2022.S1901-Data.csv
ACSST1Y2023.S1901-Column-Metadata.csv
ACSST1Y2023.S1901-Data.csv
ACSST1Y2024.S1901-Column-Metadata.csv
ACSST1Y2024.S1901-Data.csv


In [ ]:
# Load 2024 ACSST income data file
file_path = '../data/raw/census_acs/ACSST1Y2024.S1901-Data.csv'

income_df_2024 = pd.read_csv(file_path,
    header=[0]  # Multi-row header based on what we saw
    )

# Inspect the file
print('Shape:', income_df_2024.shape)
print('\nColumn names:')
for col in income_df_2024.columns:
    print(col)

Shape: (905, 131)

Column names:
GEO_ID
NAME
S1901_C01_001E
S1901_C01_001M
S1901_C01_002E
S1901_C01_002M
S1901_C01_003E
S1901_C01_003M
S1901_C01_004E
S1901_C01_004M
S1901_C01_005E
S1901_C01_005M
S1901_C01_006E
S1901_C01_006M
S1901_C01_007E
S1901_C01_007M
S1901_C01_008E
S1901_C01_008M
S1901_C01_009E
S1901_C01_009M
S1901_C01_010E
S1901_C01_010M
S1901_C01_011E
S1901_C01_011M
S1901_C01_012E
S1901_C01_012M
S1901_C01_013E
S1901_C01_013M
S1901_C01_014E
S1901_C01_014M
S1901_C01_015E
S1901_C01_015M
S1901_C01_016E
S1901_C01_016M
S1901_C02_001E
S1901_C02_001M
S1901_C02_002E
S1901_C02_002M
S1901_C02_003E
S1901_C02_003M
S1901_C02_004E
S1901_C02_004M
S1901_C02_005E
S1901_C02_005M
S1901_C02_006E
S1901_C02_006M
S1901_C02_007E
S1901_C02_007M
S1901_C02_008E
S1901_C02_008M
S1901_C02_009E
S1901_C02_009M
S1901_C02_010E
S1901_C02_010M
S1901_C02_011E
S1901_C02_011M
S1901_C02_012E
S1901_C02_012M
S1901_C02_013E
S1901_C02_013M
S1901_C02_014E
S1901_C02_014M
S1901_C02_015E
S1901_C02_015M
S1901_C02_016E
S1901_C02_

In [28]:
income_df_2024.head(3)

,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001M,S1901_C01_002E,S1901_C01_002M,S1901_C01_003E,S1901_C01_003M,S1901_C01_004E,S1901_C01_004M,S1901_C01_005E,S1901_C01_005M,S1901_C01_006E,S1901_C01_006M,S1901_C01_007E,S1901_C01_007M,S1901_C01_008E,S1901_C01_008M,S1901_C01_009E,S1901_C01_009M,S1901_C01_010E,S1901_C01_010M,S1901_C01_011E,S1901_C01_011M,S1901_C01_012E,S1901_C01_012M,S1901_C01_013E,S1901_C01_013M,S1901_C01_014E,S1901_C01_014M,S1901_C01_015E,S1901_C01_015M,S1901_C01_016E,S1901_C01_016M,S1901_C02_001E,S1901_C02_001M,S1901_C02_002E,S1901_C02_002M,S1901_C02_003E,S1901_C02_003M,S1901_C02_004E,S1901_C02_004M,S1901_C02_005E,S1901_C02_005M,S1901_C02_006E,S1901_C02_006M,S1901_C02_007E,S1901_C02_007M,S1901_C02_008E,S1901_C02_008M,S1901_C02_009E,S1901_C02_009M,S1901_C02_010E,S1901_C02_010M,S1901_C02_011E,S1901_C02_011M,S1901_C02_012E,S1901_C02_012M,S1901_C02_013E,S1901_C02_013M,S1901_C02_014E,S1901_C02_014M,S1901_C02_015E,S1901_C02_015M,S1901_C02_016E,S1901_C02_016M,S1901_C03_001E,S1901_C03_001M,S1901_C03_002E,S1901_C03_002M,S1901_C03_003E,S1901_C03_003M,S1901_C03_004E,S1901_C03_004M,S1901_C03_005E,S1901_C03_005M,S1901_C03_006E,S1901_C03_006M,S1901_C03_007E,S1901_C03_007M,S1901_C03_008E,S1901_C03_008M,S1901_C03_009E,S1901_C03_009M,S1901_C03_010E,S1901_C03_010M,S1901_C03_011E,S1901_C03_011M,S1901_C03_012E,S1901_C03_012M,S1901_C03_013E,S1901_C03_013M,S1901_C03_014E,S1901_C03_014M,S1901_C03_015E,S1901_C03_015M,S1901_C03_016E,S1901_C03_016M,S1901_C04_001E,S1901_C04_001M,S1901_C04_002E,S1901_C04_002M,S1901_C04_003E,S1901_C04_003M,S1901_C04_004E,S1901_C04_004M,S1901_C04_005E,S1901_C04_005M,S1901_C04_006E,S1901_C04_006M,S1901_C04_007E,S1901_C04_007M,S1901_C04_008E,S1901_C04_008M,S1901_C04_009E,S1901_C04_009M,S1901_C04_010E,S1901_C04_010M,S1901_C04_011E,S1901_C04_011M,S1901_C04_012E,S1901_C04_012M,S1901_C04_013E,S1901_C04_013M,S1901_C04_014E,S1901_C04_014M,S1901_C04_015E,S1901_C04_015M,S1901_C04_016E,S1901_C04_016M,Unnamed: 130
0,Geography,Geographic Area Name,Estimate!!Households!!Total,Margin of Error!!Households!!Total,"Estimate!!Households!!Total!!Less than $10,000",Margin of Error!!Households!!Total!!Less than ...,"Estimate!!Households!!Total!!$10,000 to $14,999","Margin of Error!!Households!!Total!!$10,000 to...","Estimate!!Households!!Total!!$15,000 to $24,999","Margin of Error!!Households!!Total!!$15,000 to...","Estimate!!Households!!Total!!$25,000 to $34,999","Margin of Error!!Households!!Total!!$25,000 to...","Estimate!!Households!!Total!!$35,000 to $49,999","Margin of Error!!Households!!Total!!$35,000 to...","Estimate!!Households!!Total!!$50,000 to $74,999","Margin of Error!!Households!!Total!!$50,000 to...","Estimate!!Households!!Total!!$75,000 to $99,999","Margin of Error!!Households!!Total!!$75,000 to...","Estimate!!Households!!Total!!$100,000 to $149,999","Margin of Error!!Households!!Total!!$100,000 t...","Estimate!!Households!!Total!!$150,000 to $199,999","Margin of Error!!Households!!Total!!$150,000 t...","Estimate!!Households!!Total!!$200,000 or more","Margin of Error!!Households!!Total!!$200,000 o...",Estimate!!Households!!Median income (dollars),Margin of Error!!Households!!Median income (do...,Estimate!!Households!!Mean income (dollars),Margin of Error!!Households!!Mean income (doll...,Estimate!!Households!!PERCENT ALLOCATED!!House...,Margin of Error!!Households!!PERCENT ALLOCATED...,Estimate!!Households!!PERCENT ALLOCATED!!Famil...,Margin of Error!!Households!!PERCENT ALLOCATED...,Estimate!!Households!!PERCENT ALLOCATED!!Nonfa...,Margin of Error!!Households!!PERCENT ALLOCATED...,Estimate!!Families!!Total,Margin of Error!!Families!!Total,"Estimate!!Families!!Total!!Less than $10,000",Margin of Error!!Families!!Total!!Less than $1...,"Estimate!!Families!!Total!!$10,000 to $14,999","Margin of Error!!Families!!Total!!$10,000 to $...","Estimate!!Families!!Total!!$15,000 to $24,999","Margin of Error!!Families!!Total!!$15,000 to $...","Estimate!!Families!!Total!!$25,000 to $34,999","Margin of Error!!Families!!Total!!$25,000 to

The median household income column is identified as 'S1901_C01_012E' 
(Estimate!!Households!!Median income (dollars)).

The Data file contains 904 rows and 131 columns, representing counties 
across all 50 states. California counties can be filtered using the GEO_ID 
column, which begins with '0500000US06' for all 58 California counties.

Note: the Data file uses a single-row header (header=0), unlike CHR which 
required a two-row MultiIndex header. The Column-Metadata file was used as 
a reference to identify the correct column code.

Suppression codes '(X)' and 'N' were observed in the data and will require 
handling in the cleaning notebook.

2020 is absent from the ACS 1-year estimates due to COVID-19 data collection 
disruptions — this is a known gap in the analysis.

In [29]:
# loop checking for income data column 'S1901_C01_012E' in each file year              
files = os.listdir('../data/raw/census_acs')
for f in sorted(files):
    if 'Data' in f:
        file_path = '../data/raw/census_acs/' + f
        df_year = pd.read_csv(file_path, header=0)
        if 'S1901_C01_012E' in df_year.columns:
            print('Found in:', f)
        else:
            print('MISSING in:', f)                

Found in: ACSST1Y2016.S1901-Data.csv
Found in: ACSST1Y2017.S1901-Data.csv
Found in: ACSST1Y2018.S1901-Data.csv
Found in: ACSST1Y2019.S1901-Data.csv
Found in: ACSST1Y2021.S1901-Data.csv
Found in: ACSST1Y2022.S1901-Data.csv
Found in: ACSST1Y2023.S1901-Data.csv
Found in: ACSST1Y2024.S1901-Data.csv


Column naming consistency was checked in all 8 target files. The convention of naming the estimated household income variable, 'S1901_C01_012E' has been verified in all years. 
The year 2020 is missing the reporting file due to disruptions arising from the Covid 19 pandemic. This is a noted data limitation.
